In [1]:
import csv
import time,json,pandas as pd,re
from funcs import transformMasterList

In [2]:
finO = open("/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv",encoding="latin")
finT = open("/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.csv")

readerO = csv.DictReader(finO,delimiter="\t")
readerT = csv.DictReader(finT)

allO = []
allT = []
for row in readerO:
    allO.append(row)

for row in readerT:
    allT.append(row)

## FUnction

In [ ]:
import os,sys,json

bic_etl_home = os.getenv('bic_etl_home')
## Add the bic_etl/general/script directory to path 
sys.path.insert(0, os.path.join(bic_etl_home, 'general', 'scripts'))

import custom_log,pandas as pd,csv
from jcLibNew import checkMissing
def transformMasterList(dirTarget,inFiles,outFile,inFileOpts,logger2,dictFile,delimiter,w4x4):
    colOrder=['farmProduct', 'debtorName', 'debtorId', 'counties', 'cropYear', 'debtorAddress', 'additionalDebtors', 'additionalDebtorId', 
              'securedParty', 'assignee', 'additionalFarmProduct', 'recordIdVerbatim', 'recordId', 'recordIdDate', 'recordCountyFilingId', 
              'recordCountyFiling', 'amendmentIdVerbatim', 'amendmentId']

# Get src-trns dictionary
    dictFile=open(f'{dirTarget}/{dictFile}')
    dictLookup = json.load(dictFile)
    dictFile.close()
    t4x4=list(dictLookup.keys())[0]
    a=dictLookup[t4x4].values()
# Get the transform columns
    trnsCols = [dct['xref'] for dct in a]
    xrefs = {k:v['xref'] for k,v in dictLookup[t4x4].items() if not v['dupe']}
    
    with open(f'{dirTarget}/{outFile}','w') as fout:
        for nfile,inFile in enumerate(inFiles):
            badColsDat=[]
            badColsDict=[]
            
            opts = inFileOpts[nfile]
            if len(opts) == 0:
                op = open(f'{dirTarget}/{inFile}','r')
                dReader = csv.DictReader(op,delimiter=delimiter)
            else:
                d = {k.strip(): v.strip() for k, v in [opts.split('=', 1)]}
                op = open(f'{dirTarget}/{inFile}','r',**d)
                dReader = csv.DictReader(op,delimiter=delimiter)
                

            logger2.info(f"Processing File {inFile}",extra={"s4x4":w4x4})
## Set up Header
            datCols=dReader.fieldnames
            print("datCols ",datCols)

            
## remove utf-8 encoding Byte Order Marks if they occur...
            # header=header.replace("\uFEFF","")
            # header=header.split(delimiter)
           
            headerNew=""

# Create Output Header AND Check Transformed Expected vs Dictionary (this step dataset specific..some wont need this)
            badTrnsExpNotData=[]
            for nn,col in enumerate(colOrder):
                col=col.strip().replace('"','')
                if col in trnsCols:
                    headerNew+=f"{col}{delimiter}"
                else:
                    badTrnsExpNotData.append(col)
                    print(f"BAD COL :{col}:")
            headerNew=headerNew.strip(delimiter)

            if len(badTrnsExpNotData) > 0:
                for col in badTrnsExpNotData:
                    logger2.error(f"Transform Column in DATA not in DICTIONARY :{col}:",extra={"s4x4":w4x4})
                logger2.error(f"Transform Failed....Transform Failed",extra={"s4x4":w4x4})
                sys.exit(1)

# Check SOURCE Dictionary columns vs actual Data columns
            badSrcDictNotData=[]
            for col,dct in xrefs.items():
                col=col.strip()
                if col not in datCols and not dct['dupe']:
                    badSrcDictNotData.append(col)
                    
            if len(badSrcDictNotData) > 0:
                for col in badSrcDictNotData:
                    logger2.error(f"Column in DICTIONARY NOT Found in DATA:{col}:",extra={"s4x4":w4x4})
                logger2.error(f"Transform Failed....Transform Failed",extra={"s4x4":w4x4})
                sys.exit(1)
                
# Check actual SOURCE Data columns vs Dictionary columns
            badSrcDataNotDict=[]
            for col in datCols:
                col=col.strip()
                if col not in xrefs:
                    badSrcDataNotDict.append(col)
                     
            if len(badSrcDataNotDict) > 0:
                for col in badSrcDataNotDict:
                    logger2.error(f"Column in DATA not in DICTIONARY :{col}:",extra={"s4x4":w4x4})
                logger2.error(f"Transform Failed....Transform Failed",extra={"s4x4":w4x4})
                sys.exit(1)


            
            if nfile == 0:  # only write the header for the first input file
                fout.write(f"{headerNew}\n")
                hist={}

#  Process Input File
            badTrnsDataNotDict=[]
            newRow={}
            nrows=0
            for row in dReader:
                nrows+=1
                newRow={xrefs[k]:v  for k,v in row.items()}
                # recordIdDate', 'recordCountyFilingId','recordCountyFiling', 'amendmentIdVerbatim', 'amendmentId'
                addDebtors, addAmendId, recId, recDate,countyId,countyName=transformMasterListAdd(row,logger2,w4x4)
                newRow['additionalDebtorId']=addDebtors
                newRow['amendmentId']=addAmendId
                newRow['recordId']=recId
                newRow['recordIdDate']=recDate
                newRow['recordCountyFilingId']=countyId
                newRow['recordCountyFiling']=countyName
                string=""
                for col in colOrder:
                    string+=f"{newRow[col]}{delimiter}"
                # Create Output Header AND Check Transformed Expected vs Dictionary (this step dataset specific..some wont need this)
                if nrows < 2: 
                    badTrnsExpNotData=[]
                    for col in newRow.keys():
                        col=col.strip()
                        if col not in trnsCols:
                            badTrnsDataNotDict.append(col)
                    if len(badTrnsDataNotDict): 
                        print("XREF VALS ",xrefs.values())
                        for col in badTrnsDataNotDict: 
                            logger2.error(f"Transform Column in DATA not in DICTIONARY :{col}:",extra={"s4x4":w4x4})
                        logger2.error(f"Transform Failed....Transform Failed",extra={"s4x4":w4x4})
                        sys.exit(1)

                string.strip(delimiter)
                fout.write(f"{string}\n")
                hist = checkMissing(string,hist,delimiter)            
            op.close()
    fout.close()




def transformMasterListAdd(row,logger2,w4x4):
#  Additional Debtors
    addDebtors=""
    if len(row['Additional Debtors']) > 0: 
        
  #     print( row['Additional Debtors'])
        spl = row['Additional Debtors'].split(".")
        cntIds = row['Additional Debtors'].count("ID #")
        cntNoIds = row['Additional Debtors'].count("NO ID #")
        cntPds = row['Additional Debtors'].count("#")
        naddDebtors=0
        for val in spl:
            indx=val.find("ID #")
            if indx > -1:
               db=val[indx+5:].strip()
               if len(db) > 0: 
                   addDebtors+=f"{db};"
                   naddDebtors+=1
        addDebtors=addDebtors.strip(";")
        if (naddDebtors != cntIds or naddDebtors != cntPds and cntIds != cntNoIds+naddDebtors) :
            logger2.debug(f"addiotionalDebtorID missmatch...Count of debtors does not meet expected count: # Debtors found {naddDebtors}  Expected Count: {cntIds}  String: {row['Additional Debtors']}",extra={"s4x4":w4x4})
            print("Additional Debtors MISMATCH ",naddDebtors,cntIds,cntPds,cntNoIds)
          


#  Amendment ID
    addAmendId=""
    if len(row['Amendment ID #(s)']) > 0: 
        spl=row['Amendment ID #(s)'].split(";")
       
        for val in spl:
            spl2=val.split("-")
            aid=spl2[0].split("(")[0]
            aid=aid.replace("'","").strip()
            addAmendId+=f"{aid};"
        addAmendId=addAmendId.rstrip(";")


#  Record ID #
    countyId=""
    countyName=""
    recId=""
    recDate=""
    if len(row['Record Id #']) > 0:
        spl= row['Record Id #'].split("-")
        if len(spl) > 2:
            spl2 = spl[0].split("(")
            recId=spl2[0].strip()
            countyId = spl2[1].strip()
    
            countyName=spl[1].strip().strip(")")
            recDate = spl[2].strip()
        elif len(spl) == 2:
            recId=spl[0].strip()
            recDate=spl[1].strip()
            countyId=""
            countyName=""
    
    return addDebtors,addAmendId,recId,recDate,countyId,countyName




dirTarget="/home/joe/bic_etl/cdos/business/business"
inFiles=["data_source/masterlist.tsv"]
outFile="data_transformed/masterlist.tsv"
inFileOpts=""
delimiter="\t"

w4x4="ej2c-jkvh"

logger2 = custom_log.setupNew("Master List in Colorado")
logger2.info(f"Starting: Master List in Colorado",extra={"s4x4":w4x4})

ndebts=0
naddAmend=0
nrecId=0
nrecDate=0
ncountyId=0
ncountyName=0
inFileOpts=["encoding=latin"]
function_name = "transformMasterList"
dictFile="defs/masterlist_ej2c-jkvh_src_trns_xrefs.json"

def getSrcTrnsDict(dirTarget,dictFile): 
    dictFile=open(f'{dirTarget}/{dictFile}')
    dictLookup = json.load(dictFile)
    dictFile.close()
    fieldXrefs={srcField:refs['xref'] for srcField,refs in dictLookup[w4x4].items() if not refs['dupe']}
    return fieldXrefs


if function_name not in globals():
    print(f"Function {function_name} NOT FOUND... Exiting")
    exit(0)



globals()[function_name](dirTarget,inFiles,outFile,inFileOpts,logger2,dictFile,delimiter,w4x4)

#print(f"{len(allO)}\n# Debtors   : {ndebts} {100*ndebts/len(allO)}\n# Amend     : {naddAmend} {naddAmend/len(allO)*100}\n# Rec ID    : {nrecId} {nrecId/len(allO)*100}\n# Rec Date  : {nrecDate} {nrecDate/len(allO)*100}\n# County ID : {ncountyId} {ncountyId/len(allO)*100}\n# County NM : {ncountyName} {ncountyName/len(allO)*100}")

{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List in Colorado","4x4": "ej2c-jkvh", "pid": "793", "level": "20", "msg": "Starting: Master List in Colorado", "time": "2025-01-15 15:36:58,242"}
{"name": "Master List 

In [23]:
def ff(**kwargs):
    print(kwargs["b"]) 

a=10
b=12
c=13

ff(a=a,b=b,c=c)

12


In [5]:
srting="BAKER, CHARLES ROERT; NO ID #. BAKER, MARLA RAE; NO ID #. BAKER, MARLA; ID #: 1012023132. CHARLES, BAKER; ID #: 4081032024"
print(len(srting.split("NO ID #")))
print(len(srting.split("ID #")))
      

3
5


## Check Results

In [12]:
fileOld = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.csv"
fileNew = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv"
fileOrg = "/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv"

dirTarget="/home/joe/bic_etl/cdos/business/business"
inFiles=["data_source/masterlist.tsv"]
outFile="data_transformed/masterlist.tsv"
inFileOpts=""
delimiter="\t"
dictFile="defs/masterlist_ej2c-jkvh_src_trns_xrefs.json"


finOld = open(fileOld)
finNew = open(fileNew)
finOrg = open(fileOrg,encoding="latin")


readerOld = csv.DictReader(finOld)
readerNew = csv.DictReader(finNew,delimiter="\t")
readerOrg = csv.DictReader(finOrg,delimiter="\t")


recsOld=[]
recsNew=[]
recsOrg=[]

for row in readerOld:
    recsOld.append(row)

for row in readerNew:
    recsNew.append(row)

for row in readerOrg:
    recsOrg.append(row)


print("DIRS ",dirTarget,dictFile)
# Get src-trns dictionary
dictFileFin=open(f'{dirTarget}/{dictFile}')
dictLookup = json.load(dictFileFin)
dictFileFin.close()
t4x4=list(dictLookup.keys())[0]
a=dictLookup[t4x4].values()
# Get the transform columns
trnsCols = [dct['xref'] for dct in a]
#xrefs = {v['xref']:k for k,v in dictLookup[t4x4].items() if not v['dupe']}

xrefs={}
for origCol,dct in dictLookup[t4x4].items():
    trnsCol = dct['xref']
    if dict['dupe']:
        origCol=re.sub("_\d+","",origCol)
        print(trnsCol,origCol)
    xrefs[trnsCol]=origCol
    

DIRS  /home/joe/bic_etl/cdos/business/business defs/masterlist_ej2c-jkvh_src_trns_xrefs.json
recordCountyFilingId Record Id #
recordIdDate Record Id #
recordCountyFiling Record Id #
assignee Assignee(s)
cropYear Crop Year(s)
counties County(ies)
recordIdVerbatim Record Id #
additionalDebtors Additional Debtors
additionalDebtorId Additional Debtors
amendmentIdVerbatim Amendment ID #(s)
recordId Record Id #
securedParty Secured Party(ies)
debtorAddress Debtor Address
debtorId Debtor ID #
debtorName Debtor Name
amendmentId Amendment ID #(s)
additionalFarmProduct Additional Farm Product(s)
farmProduct Farm Product
farmProductDescription Description of Farm Product(s)


In [13]:
# Org  -> Orignal Source File
# Old  -> Old Transformed File
# New  -> New Transformed file

# xrefs[trnsCol]=origCol
hist={}
bads={}
spaces = ["counties","additionalFarmProduct"]
caps = ["recordCountyFiling"]
for nn,rowOrg in enumerate(recsOrg):
    rowNew=recsNew[nn]
    rowOld=recsOld[nn]

    for keyOld,valOld in rowOld.items():
        valNew = rowNew[keyOld]
        keyOrg = xrefs[keyOld]
        valOrg=rowOrg[keyOrg]
        if keyOld not in hist:
            hist[keyOld]={}
            hist[keyOld]['checked']=0
            hist[keyOld]['good']=0
            hist[keyOld]['bad']=0
            bads[keyOld]={}
        if keyOld in spaces:
            valNew=valNew.replace(" ","")
            valOld=valOld.replace(" ","")

        if keyOld in caps:
            valNew=valNew.upper()
            valOld=valOld.upper()
            
            
        hist[keyOld]['checked']+=1
   
     
        if valOld != valNew:
            hist[keyOld]['bad']+=1
            if valOld not in bads[keyOld]:
                bads[keyOld][valOld]={}
                bads[keyOld][valOld]["Count"]=0
                
            bads[keyOld][valOld]["Trans"]=valNew
            
            bads[keyOld][valOld]["Count"]+=1
            
            if keyOld in xrefs:
                colOrg = xrefs[keyOld]
                bads[keyOld][valOld]["Orig"]=recsOrg[nn][colOrg]
            else:
                bads[keyOld][valOld]["Orig"]="Crapper"
            
            
        else:
            hist[keyOld]['good']+=1
            
    
        

In [14]:
print(f"{'good':25s} {'checked':7s}  {'  good':6s} {'    bad':6s}")
for key,dct in hist.items():
    print(f"{key:25s} {dct['checked']:7d}  {dct['good']:6d}  {dct['bad']:6d}")

good                      checked    good     bad
farmProduct                 59030       3   59027
debtorName                  59030   59030       0
debtorId                    59030   59030       0
counties                    59030   59030       0
cropYear                    59030   59030       0
debtorAddress               59030   57236    1794
additionalDebtors           59030   59030       0
additionalDebtorId          59030   59030       0
securedParty                59030   58993      37
assignee                    59030    2391   56639
additionalFarmProduct       59030   59030       0
recordIdVerbatim            59030   59030       0
recordId                    59030   59030       0
recordIdDate                59030       0   59030
recordCountyFilingId        59030   59030       0
recordCountyFiling          59030   59030       0
amendmentIdVerbatim         59030   59030       0
amendmentId                 59030   59030       0


In [15]:
#key = "debtorAddress"
#key = "additionalDebtorId"
#key = "securedParty"
key = "assignee"
#key = "amendmentIdVerbatim"
key = "amendmentId"

# key = ""

dct=bads[key]
nlines=0
for old,dct2 in dct.items():
    print(f"{key:25s} :\nCOUNT: {dct2['Count']:5d}\n OLD:{old:30s}\nTran:{dct2['Trans']:30s}\nOrig:{dct2['Orig']:30s}\n")
    nlines+=1
    if nlines > 20:
        break

xrefs

In [ ]:
print(allO[0].keys())
print(allT[0].keys())


In [ ]:
for nn in range(5,20):
    print(nn)
    print(allO[nn]['Record Id #'])
    print("")
    spl= allO[nn]['Record Id #'].split("-")
    if len(spl) > 2:
        spl2 = spl[0].split("(")
        id=spl2[0].strip()
        countyId = spl2[1].strip()

        countyName=spl[1].strip().strip(")")
        countyDate = spl[2].strip()
    elif len(spl) == 2:
        id=spl[0].strip()
        countyDate=spl[1].strip()
        countyId=""
        countyName=""
    print(allT[nn]['recordIdVerbatim'])
    print("")
    print(allT[nn]['recordId'])
    print(id)
    print("")
    print(allT[nn]['recordIdDate'])
    print(countyDate)
    print("")
    print(allT[nn]['recordCountyFilingId'])
    print(countyId)
    print("")
    print(allT[nn]['recordCountyFiling'])
    print(countyName)
    print("----------")


In [ ]:
spl=allO[3]['Amendment ID #(s)'].split(";")
addAmendId=""
for val in spl:
    spl2=val.split("-")
    aid=spl2[0].replace("'","").strip()
    addAmendId+=f"{aid},"
addAmendId=addAmendId.rstrip(",")
print(addAmendId)

In [ ]:
allO[0].keys()

In [ ]:
for nn,oldrec in enumerate(recsOld):
    newrec = recsNew[nn]
    print(oldrec,"\n",newrec)
    print()
    if nn > 5:
        break

In [28]:
fileOld = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.csv"
fileNew = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv"
fileOrg = "/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv"

finOld=open(fileOld)
finNew=open(fileNew)
finOrg=open(fileOrg,encoding="latin")

recsOld=finOld.readlines()
recsNew=finNew.readlines()
recsOrg=finOrg.readlines()



In [18]:
for nn,rowOrg in enumerate(recsOrg):
    rowNew=recsNew[nn]
    rowOld=recsOld[nn]

    keyOld="amendmentId"
    keyOrg=xrefs[keyOld]
    valNew = rowNew[keyOld]
    valOld = rowOld[keyOld]
    
    keyOrg = xrefs[keyOld]
    valOrg=rowOrg[keyOrg]
    print(valOrg)
    print(valNew)
    print(valOld)
    print("--")
    if nn > 10:
        break

20232038290 - 04/20/2023
20232038290
20232038290
--



--



--
20162024080 - 03/17/2016; 20212013535 - 02/11/2021
20162024080;20212013535
20162024080;20212013535
--
20242065504 - 07/18/2024
20242065504
20242065504
--
2002J048916 (KIT CARSON) - 02/13/1997; 2010F020640 - 03/10/2010; 20212114843 - 11/19/2021; 20232075277 - 08/03/2023; 2007F008442 - 01/25/2007; 20172022331 - 03/09/2017; 2002F017738 - 02/18/2002; 2010F072860 - 10/08/2010; 2012F006182 - 01/31/2012
2002J048916;2010F020640;20212114843;20232075277;2007F008442;20172022331;2002F017738;2010F072860;2012F006182
2002J048916;2010F020640;20212114843;20232075277;2007F008442;20172022331;2002F017738;2010F072860;2012F006182
--



--
2004F096326 - 08/31/2004; 20152081285 - 09/01/2015; 2002J052692 (YUMA) - 06/24/1997; 2010F064677 - 08/26/2010; 2007F075867 - 07/24/2007; 2003J000182 (20027 - YUMA) - 08/12/1985; 20202117328 - 10/15/2020; 20122076826 - 12/12/2012; 2004F096319 - 08/31/2004; 2003J000183 (20027 - YUMA) - 08/28/1990; 2002J052688 (Y

In [ ]:
for nn,oldrec in enumerate(recsOld):
    newrec=recsNew[nn]
    orgrec=recsOrg[nn]
    print("Orig: ",orgrec[-40:])
    print(" OLD: ",oldrec[-40:])
    print(" NEW: ",newrec[-40:])
    print()
    if nn > 10:
        break
    
    

In [21]:
fileNew = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv"
finNew = open(fileNew)

readerNew = csv.DictReader(finNew,delimiter="\t")

new=[]
for line in readerNew:
   new.append(line)

fileOrg = "/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv"
finOrg = open(fileOrg,encoding="latin")

readerOrg = csv.DictReader(finOrg,delimiter="\t")

org=[]
for line in readerOrg:
   org.append(line)

In [22]:
readerNew.fieldnames

['farmProduct',
 'farmProductDescription',
 'debtorName',
 'debtorId',
 'counties',
 'cropYear',
 'debtorAddress',
 'additionalDebtors',
 'additionalDebtorId',
 'securedParty',
 'assignee',
 'additionalFarmProduct',
 'recordIdVerbatim',
 'recordId',
 'recordIdDate',
 'recordCountyFilingId',
 'recordCountyFiling',
 'amendmentIdVerbatim',
 'amendmentId']

In [23]:
nrec=0
import time
for row in new:
    print(row['amendmentId'])
    time.sleep(.5)
    if nrec > 100:
        break

20232038290


20162024080;20212013535
20242065504
2002J048916;2010F020640;20212114843;20232075277;2007F008442;20172022331;2002F017738;2010F072860;2012F006182

2004F096326;20152081285;2002J052692;2010F064677;2007F075867;2003J000182;20202117328;20122076826;2004F096319;2003J000183;2002J052688;2010F064567;2003J000184;2005F069529;2010F064669
2004F096326;20152081285;2002J052692;2010F064677;2007F075867;2003J000182;20202117328;20122076826;2004F096319;2003J000183;2002J052688;2010F064567;2003J000184;2005F069529;2010F064669
20192068942;20152000631;20242060358


20232036695


20192071875;20232030716;20232007933
2002J053455;2002F059649;20222037687;20122022012;2007F062942;20172030263;2002J017507;20232079643
2002J053455;2002F059649;20222037687;20122022012;2007F062942;20172030263;2002J017507;20232079643
2002J053455;2002F059649;20222037687;20122022012;2007F062942;20172030263;2002J017507;20232079643


KeyboardInterrupt: 

In [2]:
dfC = pd.read_csv("https://data.colorado.gov/resource/ej2c-jkvh.csv")

In [6]:
set(readerNew.fieldnames) - set(dfC.columns)

{'additionalDebtorId',
 'additionalDebtors',
 'additionalFarmProduct',
 'amendmentId',
 'amendmentIdVerbatim',
 'cropYear',
 'debtorAddress',
 'debtorId',
 'debtorName',
 'farmProduct',
 'recordCountyFiling',
 'recordCountyFilingId',
 'recordId',
 'recordIdDate',
 'recordIdVerbatim',
 'securedParty'}

In [8]:
set(dfC.columns)-set(readerNew.fieldnames.lower)

AttributeError: 'list' object has no attribute 'lower'

In [24]:
newTr = [col.lower() for col in readerNew.fieldnames]

In [25]:
set(dfC.columns)-set(newTr)

set()

In [26]:
set(newTr)-set(dfC.columns)


{'farmproductdescription'}

In [14]:
len(dfC.columns)

18

In [15]:
len(newTr)

18

In [20]:
readerOrg.fieldnames



['Farm Product',
 'Debtor Name',
 'Debtor ID #',
 'County(ies)',
 'Crop Year(s)',
 'Debtor Address',
 'Additional Debtors',
 'Secured Party(ies)',
 'Assignee(s)',
 'Additional Farm Product(s)',
 'Description of Farm Product(s)',
 'Record Id #',
 'Amendment ID #(s)']

In [19]:
newTr

['farmproduct',
 'debtorname',
 'debtorid',
 'counties',
 'cropyear',
 'debtoraddress',
 'additionaldebtors',
 'additionaldebtorid',
 'securedparty',
 'assignee',
 'additionalfarmproduct',
 'recordidverbatim',
 'recordid',
 'recordiddate',
 'recordcountyfilingid',
 'recordcountyfiling',
 'amendmentidverbatim',
 'amendmentid']

In [27]:
dfT = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv",delimiter="\t")

In [28]:
dfT.columns

Index(['farmProduct', 'farmProductDescription', 'debtorName', 'debtorId',
       'counties', 'cropYear', 'debtorAddress', 'additionalDebtors',
       'additionalDebtorId', 'securedParty', 'assignee',
       'additionalFarmProduct', 'recordIdVerbatim', 'recordId', 'recordIdDate',
       'recordCountyFilingId', 'recordCountyFiling', 'amendmentIdVerbatim',
       'amendmentId'],
      dtype='object')

In [29]:
dfT['farmProductDescription'].value_counts()

farmProductDescription
FIX LAND CATTLE CO                                6450
BREIDENBACH BROS INC                               392
WESTERN SUGAR COOPERATIVE (THE)                    355
ROCKY MOUNTAIN SUGAR GROWERS COOPERATIVE           351
NEB BROTHERS                                       318
                                                  ... 
ROCKY MOUNTAIN SUGAR GROWERS                         1
ROCKY MOUNTAIN SUGAR GROWERS COOOPERATIVE            1
ROCKY MOUNTAIN SUGAR GROWERS COOPERAITVE             1
ROCKY MOUNTAIN SUGAR GROWERS COOPERATIVE (THE)       1
WESTERN SUGAR GROWERS COOPERATIVE (THE)              1
Name: count, Length: 9838, dtype: int64

In [30]:
dfT['amendmentId'].value_counts()

Series([], Name: count, dtype: int64)

In [31]:
new[0]

{'farmProduct': 'All Field Crops',
 'farmProductDescription': '',
 'debtorName': '100X FARM',
 'debtorId': '6066242134',
 'counties': 'Boulder; Larimer; Weld',
 'cropYear': 'ALL',
 'debtorAddress': '1716 MAIN ST, LONGMONT, CO, 80501',
 'additionalDebtors': 'BLOOMSTEADS; ID #: 6126022194. CULINARY FARMSTEADS; ID #: 6216032194. MANNINGLEBEK GROUP LLC; ID #: 6016132034. MANNINGLEBEK, CHRISTINA SHERRILL; ID #: 2016136039. MANNINGLEBEK, JOCHEN; ID #: 8018135101',
 'additionalDebtorId': '6126022194;6216032194;6016132034;2016136039;8018135101',
 'securedParty': 'FARM SERVICE AGENCY for UNITED STATES OF AMERICA, 57 W. BROMLEY LANE, BRIGHTON, CO, 80601',
 'assignee': 'NONE',
 'additionalFarmProduct': '',
 'recordIdVerbatim': '20222046557 - 05/05/2022',
 'recordId': '20222046557',
 'recordIdDate': '05/05/2022',
 'recordCountyFilingId': '',
 'recordCountyFiling': '',
 'amendmentIdVerbatim': '20232038290 - 04/20/2023',
 'amendmentId': '20232038290',
 None: ['']}

In [ ]:
for row in new:
    val=row['amendmentId']
    for c in val:
        i=ord(c)
        if i not in hist:
            hist[i]+=1

In [43]:
fileNew = "/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv"
finNew = open(fileNew)
allNew=finNew.readlines()

In [44]:
hist={}
for line in allNew:
    spl=line.split("\t")
    nn=len(spl)
    if nn not in hist:
        hist[nn]=0
    hist[nn]+=1
    # if nn != 20:
    #     print(line)
    #     for mm,v in enumerate(spl):
    #         print(mm+1,v)

In [45]:
hist

{19: 59031}

In [39]:
line=allNew[1]
for nn,val in enumerate(line.split("\t")):
    print(nn+1,val)

1 All Field Crops
2 
3 100X FARM
4 6066242134
5 Boulder; Larimer; Weld
6 ALL
7 1716 MAIN ST, LONGMONT, CO, 80501
8 BLOOMSTEADS; ID #: 6126022194. CULINARY FARMSTEADS; ID #: 6216032194. MANNINGLEBEK GROUP LLC; ID #: 6016132034. MANNINGLEBEK, CHRISTINA SHERRILL; ID #: 2016136039. MANNINGLEBEK, JOCHEN; ID #: 8018135101
9 6126022194;6216032194;6016132034;2016136039;8018135101
10 FARM SERVICE AGENCY for UNITED STATES OF AMERICA, 57 W. BROMLEY LANE, BRIGHTON, CO, 80601
11 NONE
12 
13 20222046557 - 05/05/2022
14 20222046557
15 05/05/2022
16 
17 
18 20232038290 - 04/20/2023
19 20232038290
20 

